# Candidate recovery metric: sleep duration and efficiency

Sleep is the wearable signal patients themselves most associate with recovery, and it is recorded
passively overnight rather than requiring the patient to move. Against that, it is the signal
most easily disrupted by things that are not recovery -- a hospital ward, analgesia, a device
removed at night -- so its face validity as a recovery measure is weaker than its coverage
suggests.

This notebook audits whether sleep is measured densely enough around surgery to support an
outcome, and whether the signal moves at all. Efficiency is carried alongside duration because
the two can disagree, and disagreement would mean two candidate metrics rather than one.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("fitbit_sleep")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

METRIC_CTE = f"""
    sleep AS (
        SELECT
            s.person_id,
            DATE_DIFF(s.sleep_date, i.index_date, DAY) AS rel_day,
            s.minute_in_bed,
            s.minute_asleep,
            SAFE_DIVIDE(s.minute_asleep, NULLIF(s.minute_in_bed, 0)) AS efficiency
        FROM `{CDR}`.sleep_daily_summary AS s
        JOIN index_surgery AS i ON i.person_id = s.person_id
        WHERE s.is_main_sleep = true
          AND s.minute_asleep IS NOT NULL
    )"""

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# 2-coverage

### Does the sleep table have the columns this notebook assumes?

`sleep_daily_summary` carries one row per sleep episode, and the `is_main_sleep` flag is what
separates a night's sleep from a nap. A missing column here changes the metric definition, not
just the query.

In [ ]:
sleep_columns = cached_gbq(
    srp_common.columns_sql(cfg, ["sleep_daily_summary"]), "sleep_daily_summary_columns"
)

if sleep_columns.empty:
    print("sleep_daily_summary not found in the CDR -- check the table name in the "
          "All of Us data dictionary for this release before going further.")
else:
    print(sleep_columns.to_string(index=False))

### How many cohort patients have any main-sleep record, and on which days?

Naps are excluded, so this counts nights rather than sleep episodes. The peri-operative dip is
the part to read: a device left at home during an admission produces exactly the pattern a
recovery metric cannot survive.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                    AS cohort_patients,
        (SELECT COUNT(DISTINCT person_id) FROM sleep)    AS patients_with_any_sleep,
        (SELECT COUNT(*) FROM sleep)                     AS sleep_nights
"""

cov = cached_gbq(coverage_sql, "sleep_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

print(f"cohort patients             : {cohort_patients:>9,}")
print(f"  ... with >=1 main sleep   : {int(cov['patients_with_any_sleep']):>9,}"
      f"   ({int(cov['patients_with_any_sleep']) / cohort_patients:6.1%} of cohort)")
print(f"main-sleep nights           : {int(cov['sleep_nights']):>9,}")

curve_sql = CTES + """
    SELECT rel_day, COUNT(DISTINCT person_id) AS n_patients
    FROM sleep
    WHERE rel_day BETWEEN -180 AND 365
    GROUP BY rel_day
    ORDER BY rel_day
"""

sleep_curve = cached_gbq(curve_sql, "sleep_coverage_curve")

srp_common.plot_rel_day_curve(
    cfg, sleep_curve, "n_patients",
    title="Patients with a Fitbit main-sleep record, by day relative to surgery",
    ylabel="patients with a sleep record", suppress=False, save_as="sleep_coverage")
srp_common.plot_rel_day_curve(
    cfg, sleep_curve, "n_patients",
    title="Patients with a Fitbit main-sleep record, by day relative to surgery",
    ylabel="patients with a sleep record", suppress=True, save_as="sleep_coverage")

### How many patients have both a usable baseline and a usable follow-up?

The same paired-window test as the other wearable metrics, so the three are directly comparable
in the scorecard. Seven nights is the minimum that makes a per-patient baseline more than noise.

In [ ]:
window_case = srp_common.window_case_sql("rel_day")

paired_sql = CTES + f"""
    , per_window AS (
        SELECT
            person_id,
            {window_case} AS window_name,
            COUNT(DISTINCT rel_day) AS n_days
        FROM sleep
        WHERE {window_case} IS NOT NULL
        GROUP BY person_id, window_name
    ),
    wide AS (
        SELECT
            person_id,
            MAX(IF(window_name = 'baseline', n_days, 0)) AS baseline_days,
            MAX(IF(window_name = 'recovery', n_days, 0)) AS recovery_days
        FROM per_window
        GROUP BY person_id
    )
    SELECT
        COUNT(*)                                            AS patients_any_window,
        COUNTIF(baseline_days >= 7)                         AS baseline_7d,
        COUNTIF(recovery_days >= 7)                         AS recovery_7d,
        COUNTIF(baseline_days >= 7 AND recovery_days >= 7)  AS paired_7d
    FROM wide
"""

paired = cached_gbq(paired_sql, "sleep_paired_windows").iloc[0]
paired_7d = int(paired["paired_7d"])

for label in ["patients_any_window", "baseline_7d", "recovery_7d", "paired_7d"]:
    n = int(paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:22s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

# 3-signal

### Do sleep duration and efficiency move after surgery, and do they agree?

Duration and efficiency are plotted on separate axes rather than one, because they are different
measures on different scales and a shared axis would misstate both. If only one of them responds
to surgery, only that one is a candidate.

In [ ]:
signal_sql = CTES + """
    SELECT
        rel_day,
        COUNT(DISTINCT person_id)                              AS n_patients,
        APPROX_QUANTILES(minute_asleep, 100)[OFFSET(50)]       AS p50_minutes_asleep,
        APPROX_QUANTILES(minute_asleep, 100)[OFFSET(25)]       AS p25_minutes_asleep,
        APPROX_QUANTILES(minute_asleep, 100)[OFFSET(75)]       AS p75_minutes_asleep,
        APPROX_QUANTILES(efficiency, 100)[OFFSET(50)]          AS p50_efficiency
    FROM sleep
    WHERE rel_day BETWEEN -60 AND 180
      AND efficiency IS NOT NULL
    GROUP BY rel_day
    ORDER BY rel_day
"""

sleep_signal = cached_gbq(signal_sql, "sleep_signal_curve")

srp_common.plot_rel_day_curve(
    cfg, sleep_signal, "p50_minutes_asleep",
    band=("p25_minutes_asleep", "p75_minutes_asleep"),
    title="Median minutes asleep (IQR band), by day relative to surgery",
    ylabel="minutes asleep", suppress=True, save_as="sleep_duration")
srp_common.plot_rel_day_curve(
    cfg, sleep_signal, "p50_efficiency",
    title="Median sleep efficiency, by day relative to surgery",
    ylabel="minutes asleep / minutes in bed", suppress=True, save_as="sleep_efficiency")

base_lo, base_hi = WINDOWS["baseline"]
for column, label in [("p50_minutes_asleep", "minutes asleep"),
                      ("p50_efficiency", "efficiency")]:
    baseline = sleep_signal.loc[
        sleep_signal["rel_day"].between(base_lo, base_hi), column].median()
    acute = sleep_signal.loc[
        sleep_signal["rel_day"].between(*WINDOWS["acute"]), column].median()
    print(f"{label:16s} baseline={baseline:8.3f}  acute={acute:8.3f}  "
          f"change={acute - baseline:+8.3f}  ({(acute - baseline) / baseline:+6.1%})")

# 4-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="sleep_daily_summary",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_sleep"]),
    patients_analysable=paired_7d,
    coverage_share=paired_7d / cohort_patients,
    horizon_days=int(WINDOWS["recovery"][1]),
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Face validity is the weak point, not coverage: ward noise, analgesia and a device "
           "left off overnight all move sleep without being recovery."),
)